In [1]:
#Task 2
# Transformer bloğunu parça parça tamamla: feedforward, residual bağlantı,
# LayerNorm. Her parçayı ekledikten sonra val loss'u kaydet. Tek tablo
# çıkar: bigram, tek head, multi-head, feedforward eklenmiş, residual
# eklenmiş, LayerNorm eklenmiş. LayerNorm'u hafta 6'da yazdığın BatchNorm1d
# ile karşılaştır: hangi eksende normalize ediyor, neden running_mean
# tutmuyor, kendi cümlelerinle yaz. (1:24:25 - 1:37:49)

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [4]:
#mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [5]:
#decoder & encoder
def encode(s):
    integers = [stoi[i] for i in s]
    return integers

def decode(l):
    lst = "".join([itos[i] for i in l])
    return lst

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)

In [7]:
#train val split with 90% 10%
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
#the way pick our batches
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) #do not want to grab latest datas like last data point as a starter
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [9]:
#self attention head as a class
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(emb_dim, head_size, bias = False)
        self.query = nn.Linear(emb_dim, head_size, bias = False)
        self.value = nn.Linear(emb_dim, head_size, bias = False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C =x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out 

In [10]:
#multi head self attention
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim = -1)
        

In [11]:
#simple mlp with ReLU 
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU())

    def forward(self, x):
        return self.net(x)               

In [12]:
#model as a class
emb_dim = 32
class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,emb_dim)   #token embeddings
        self.positional_embedding_table = nn.Embedding(block_size,emb_dim) #position embeddings
        self.slfattention_head = MultiHeadAttention(4,emb_dim//4)
        self.fddfrwd = FeedForward(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)
        
    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.positional_embedding_table(torch.arange(T,))
        x = tok_emb + pos_emb
        x = self.slfattention_head(x)
        x = self.fddfrwd(x)
        logits = self.lm_head(x) 
        
        if targets is None:
            loss =None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)         
                
        return logits, loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:,-block_size:]

            logits, loss =self(idx_cond)

            logits = logits[:,-1,:]

            probs = F.softmax(logits,dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)

In [13]:
#lets train our model
batch_size = 32
epoch = 200000

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for _ in range(epoch):

    xb,yb = get_batch("train") #first gather batch
    logits, loss = m(xb,yb) 
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f'the train loss is {loss.item()}')

the train loss is 1.8781870603561401


In [14]:
#val loss estimator
@torch.no_grad()
def val_loss_estimator():
    losses = torch.zeros(200)
    for _ in range (200):
        x,y = get_batch("val")
        logits, loss = m(x,y)
        losses[_]= loss.item()
    print(f'the validation loss is around {losses.mean()}')
val_loss_estimator()

the validation loss is around 2.105414390563965


In [15]:
#our tiny shakespeare which starts with ROMEO: haha
start = torch.tensor([encode("Romeo:")], dtype=torch.long)
out = m.generate(start, 2000)
print(decode(out[0].tolist()))

Romeo:
Bear to fulmy thems so could.

So the there, and anst weare;
Or of my. Abebase wille,
The on agefortuch; hems thould, seear he midges I have me, donow to comentst notig int Monethy that of's fornumong!

BUABE VIRCUTIO:
And:

Thet hear a challe wichavre reare be rack me, at deandy, tho kis, I prow frougs, earsin do youidess,
so untert,
I forths jierok'd the; bodiecterso Rof sta.
Timally nent of hire well he curs of let ot boude. I
hat rast's goold,
But this to the the it; you be
me the then to thernen me or dany, morfor at las: in cepark.
Be'll'd tho not, it for mesicand in his rast himplen'd his he com prade heless dower
qilike cest, eis thim tand you my here joy, for take, God,
O all you thibe elies turnese a for loon: the cains hophearty beibourd:
Yourson's in Erou.
Hery.

Wul seed ould, I Gopto sive day sawn, leodds no norme?

SB:

Refeithmy mysbe asthirst thould yest
Mi' weentull her! A shy have the to is breet isworlly.

DUCITONRY VI:
Go her nottr but.

GRUVEB:
Not of us he